# Day 1 - letters + INCLUDE keypoints (CPU only, Internet ON)
Inputs needed: `slr-code`, ASL Alphabet (https://www.kaggle.com/datasets/grassknoted/asl-alphabet) and, optionally, an ISL alphabet image dataset. The pretrained-ASL smoke test and the `asl-signs` competition are NOT needed here: they belong to Day 2 (ASL).

In [ ]:
# Code bundle: upload dist/slr_code.zip as a Kaggle Dataset named "slr-code", then add it as input.
import glob, os, shutil, subprocess, sys
src = (glob.glob("/kaggle/input/slr-code*") or glob.glob("/content/slr_code"))[0]
shutil.copytree(src, "/kaggle/working/code" if os.path.exists("/kaggle") else "/content/code", dirs_exist_ok=True)
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
CODE = f"{WORK}/code"
os.chdir(CODE)
# mediapipe 0.10.14 still has the `mp.solutions` API we use. If pip complains about numpy, run
# `!pip install "numpy<2"` and restart the session once.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe==0.10.14", "remotezip", "opencv-python-headless"])

## 1. Letters: ~300 images per letter -> hand landmarks -> classifier (minutes on CPU)
Open the *Input* panel (right sidebar), hover the dataset, click the copy-path icon, paste below. Set `ISL_IMAGES = None` if you have no ISL letter images yet - ASL still runs, ISL letters can be added later.

In [ ]:
ASL_IMAGES = "/kaggle/input/asl-alphabet"          # class-folder level is auto-detected
ISL_IMAGES = None                                    # e.g. "/kaggle/input/<your-isl-alphabet-dataset>"
for lang, root in [("asl", ASL_IMAGES), ("isl", ISL_IMAGES)]:
    if root is None:
        print("skipping", lang); continue
    !python scripts/letters.py extract --images $root --lang $lang --out $WORK/letters --per-class 300 --workers 4
    !python scripts/letters.py train --lang $lang --data $WORK/letters --out $WORK/models

## 2. INCLUDE-50 keypoints (reads only the 50 signs' videos from each Zenodo part, deletes each video right after)
Run `--list` first to check Internet/Zenodo. The second cell is the long one - use Save & Run All for it.

In [ ]:
!python scripts/include_extract.py --out $WORK/include50 --list

In [ ]:
!python scripts/include_extract.py --out $WORK/include50 --workers 8 --stride 2
!du -sh $WORK/include50

## 3. Save everything
Save Version -> Save & Run All, then Output -> **New Dataset** to keep `models/`, `letters/`, `include50/`.